# Answer lock-in in DiffusionGemma

Colab runner for the full pipeline. Requires an **A100 80GB** runtime
(Runtime -> Change runtime type -> A100).

Run the setup cells (1-5) once per session, then the pipeline cells in order.
Every GPU cell writes its output to Drive, so a runtime disconnect costs
time but never data.

**Before starting:** add `HF_TOKEN` (and optionally `GH_TOKEN`) in the key
icon in the left sidebar. Never paste tokens into a cell.

## 1. Check the GPU

Stop here if this is not an A100 with ~80GB. The 26B model will not fit otherwise.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| available', torch.cuda.is_available())

## 2. Dependencies

Colab ships torch with matching torchvision/torchaudio. Installing a
different transformers pulls no torch, but torchaudio is the package that
most often breaks the import chain after any torch-adjacent change, and it
is unused here, so it is removed rather than reconciled.

If pip reports a restart is needed, use **Runtime -> Restart session** and
resume from cell 3 (do not re-run cell 2).

In [ ]:
!pip uninstall -y -q torchaudio 2>/dev/null
!pip install -q -U 'transformers>=5.11.0' accelerate datasets matplotlib
import transformers, importlib.util
print('transformers', transformers.__version__)
assert importlib.util.find_spec('transformers.models.diffusion_gemma'), (
    'DiffusionGemma missing: transformers is too old')
print('DiffusionGemma available')

## 3. Mount Drive and link the output directories

`data/` and `results/` become symlinks into Drive, so every cached
trajectory, JSONL, and figure survives a runtime reset. This is the single
most important cell in the notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
DRIVE_ROOT = '/content/drive/MyDrive/diffusion-anchors-out'
for sub in ('data', 'results'):
    pathlib.Path(f'{DRIVE_ROOT}/{sub}').mkdir(parents=True, exist_ok=True)
print('Drive output root:', DRIVE_ROOT)

## 4. Clone the repository

Public clone over HTTPS. If the repo is private, set `GH_TOKEN` in the
secrets panel and the cell will use it.

In [ ]:
import os, subprocess
from google.colab import userdata

REPO = 'AshwathKarunakaram/diffusion-anchors'
BRANCH = 'final_clean'
WORKDIR = '/content/diffusion-anchors'

try:
    token = userdata.get('GH_TOKEN')
    url = f'https://{token}@github.com/{REPO}.git'
except Exception:
    url = f'https://github.com/{REPO}.git'

if not os.path.exists(WORKDIR):
    subprocess.run(['git', 'clone', '-b', BRANCH, url, WORKDIR], check=True)
else:
    subprocess.run(['git', '-C', WORKDIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', WORKDIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', WORKDIR, 'pull', 'origin', BRANCH], check=True)

os.chdir(WORKDIR)

# Point data/ and results/ at Drive.
for sub in ('data', 'results'):
    if os.path.islink(sub):
        os.unlink(sub)
    elif os.path.isdir(sub):
        import shutil; shutil.rmtree(sub)
    os.symlink(f'{DRIVE_ROOT}/{sub}', sub)

print('cwd:', os.getcwd())
print('data ->', os.path.realpath('data'))
print('results ->', os.path.realpath('results'))

## 5. Hugging Face authentication

The model is gated. `HF_TOKEN` must be a token on an account with access.

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import login

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
login(token=os.environ['HF_TOKEN'])
os.environ.setdefault('PYTHONPATH', '')
os.environ['PYTHONPATH'] = f"{os.getcwd()}/src:{os.environ['PYTHONPATH']}"
print('authenticated; PYTHONPATH set')

---
# Pipeline

Each cell prints elapsed time. Approximate costs on an A100 are noted per
cell; the first model load adds a few minutes of weight download.

### Resuming across sessions

Every long GPU cell takes `--resume` and skips work already cached in
Drive, so a disconnect costs only the run that was in flight. To pick up
on another day: re-run setup cells 1-5, then re-run the cell you stopped
on. Finished runs are skipped and printed as such.

`relabel_lockin_sweep.py` rebuilds the results file from the per-run cache
rather than appending, so re-running a sweep can never double-count.


## 0. Parity check (~5 min)

Confirms the instrumented denoising loop reproduces stock `generate()`.
Everything downstream depends on this, so read the verdict before continuing.
A mismatch confined to expert-routing float noise is expected and explained
in the output; a structural mismatch is not.

In [ ]:
%%time
!python src/custom_denoise.py

## 1. Behavioural sweep (~60-90 min)

19 prompt families x 10 seeds. `--overwrite` clears earlier JSONL rows;
drop it to append to an existing sweep.

In [ ]:
%%time
# --resume skips (prompt, seed) pairs already cached in Drive, so a
# disconnect costs only the run that was in flight. Drop --overwrite
# when resuming: relabel rebuilds the JSONL from the cache anyway.
!python src/generate_lockin_sweep.py --n-seeds 10 --overwrite --resume

## 2. Deepen the two mixed families to 40 seeds (~60 min)

In [ ]:
%%time
!python src/generate_lockin_sweep.py --n-seeds 30 --start-seed 10 \
    --prompt two_aces_hand --prompt increasing_three_digits --resume

## 3. Relabel from cached drafts (seconds, CPU)

Rebuilds every label with the current extractor. Safe to re-run any time.

In [ ]:
!python src/relabel_lockin_sweep.py

## 4. Lens hook alignment check (~3 min)

**Layer 29 agreement must read 1.0 at every step.** That composition is the
model's real output path, so anything else means the hooks are capturing the
wrong tensor and no downstream lens number is trustworthy. Low agreement at
layers 0 and 15 is the expected finding, not a bug.

In [ ]:
%%time
!python src/capture_lens.py --smoke

## 5. Lens capture across (layer x step) (~45 min)

In [ ]:
%%time
!python src/capture_lens.py --prompt two_aces_hand \
    --prompt increasing_three_digits --max-per-label 12 --resume

## 6. Lens figures (seconds, CPU)

In [ ]:
!python src/analyze_lens.py
from IPython.display import Image, display
import glob
for path in sorted(glob.glob('results/plots/lens/*.png')):
    print(path); display(Image(path))

## 7. Causal patch with both controls (~60 min)

The headline experiment. `--locked-donor` is the control that matters: if
state from another *locked* run rescues at the same rate as corrector state,
the claim reduces to "any coherent state resets the computation".

In [ ]:
%%time
!python src/patch_lockin.py --prompt two_aces_hand --steps 1,2 \
    --n-donors 3 --locked-donor

## 8. Point-of-no-return sweep (~45 min)

Patches at progressively later steps to find where rescue stops working.
`not_answer` also tests whether the fate lives outside the answer positions.

In [ ]:
%%time
!python src/patch_lockin.py --prompt two_aces_hand --steps 4,6,8,10 \
    --regions all,not_answer --skip-shuffle

## 9. Patch figures and exact tests (seconds, CPU)

Read every arm against the natural correct rate printed here, not against
zero: an arm sitting at that rate carries no evidence either way.

In [ ]:
!python src/analyze_patch.py
from IPython.display import Image, display
import glob
for path in sorted(glob.glob('results/plots/patch/*.png')):
    print(path); display(Image(path))

## 10. Steering hook alignment (~3 min)

Self-conditioning calls should equal denoising steps, and the hidden shape
should be (256, 2816).

In [ ]:
%%time
!python src/steer_lockin.py --smoke

## 11. Steering: rescue / random / induce (~50 min)

Per-position mode keeps the structure a pooled direction throws away. The
pooled variant already gave a negative result; running both is what makes
"causally sufficient but not low-rank" a supported claim rather than an
untested guess.

In [ ]:
%%time
!python src/steer_lockin.py --prompt two_aces_hand --mode perpos
!python src/steer_lockin.py --prompt two_aces_hand --mode pooled

## 11b. Cross-prompt transfer of the direction (~25 min, optional)

Builds the direction on one family and applies it to the other. Only
meaningful if the same-prompt steering above showed an effect.

In [ ]:
%%time
!python src/steer_lockin.py --prompt increasing_three_digits \
    --direction-from two_aces_hand --mode perpos

## 12. Doom detector (~30 min GPU, then instant CPU)

Linear probe on the step-1 state predicting eventual lock-in. Cross-prompt
AUC is the number that distinguishes a shared representation from per-prompt
memorisation. Re-run with `--train-only` to retune without touching the GPU.

In [ ]:
%%time
!python src/doom_detector.py

## 13. Expert routing (optional, exploratory)

Run the smoke cell first and read the parsed shape. If it is not
(positions, experts) with rows summing to ~1, stop: the divergence numbers
would be computed off a misparsed tensor. Nothing else depends on this.

In [ ]:
!python src/capture_routing.py --smoke

In [ ]:
%%time
!python src/capture_routing.py --prompt two_aces_hand --step 1

---
## Verify persistence

`data/` and `results/` are symlinks into Drive, so results are already
saved. This cell confirms it and prints what is there.

In [ ]:
import os, glob
print('results ->', os.path.realpath('results'))
for path in sorted(glob.glob('results/**/*', recursive=True)):
    if os.path.isfile(path):
        print(f'{os.path.getsize(path) / 1e3:9.1f} KB  {path}')
for name in ('data/lockin_sweep', 'data/lens_capture', 'data/doom_detector'):
    if os.path.isdir(name):
        print(f'{len(os.listdir(name)):9d} files  {name}/')

## Optional: commit results to the repo

`results/` is gitignored, so `-f` is required. The trajectory caches under
`data/` stay out of git — they live in Drive and are large.

In [ ]:
!git config --global user.email 'ashwath.karunakaram245@gmail.com'
!git config --global user.name 'Ashwath Karunakaram'
!git add -f results/*.jsonl results/*.json results/plots
!git commit -m 'Add pipeline results' || echo 'nothing to commit'
!git push origin final_clean